# Lab 12 - Evaluate the live Azure AI Search RAG path

## Where does an answer go wrong?

Lab 8 scored saved examples. Now run the WHO Search index and answer model together:

```text
question -> Azure AI Search -> selected context -> cited model answer
```

Check each step separately:

| Step | Question |
|---|---|
| Retrieval | Did Search find and rank the expected publication? |
| Context | Did the selected passages provide the needed evidence? |
| Answer | Are all required parts covered and cited? |
| Judging | Is the answer supported and complete? |
| Operations | How long did it take, and what did it use? |

You will run three WHO questions and use these checks to decide what needs improvement.

## Before you start

Complete Lab 3 and configure the Foundry project, model deployment, Search endpoint and index name. You need Foundry User and Search Index Data Reader access.

Replace each `...` blank before running its cell.

> Three questions demonstrate the method. They do not establish clinical validity or production reliability.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0" "azure-search-documents==12.0.0"

## 0. Connect to Search and Foundry

The next cell creates `SearchClient` for retrieval and `AIProjectClient` for answer generation and evaluation.

Both this lab and Lab 13 use `evaluation_utils.py` to calculate metrics, check citations and record usage in the same way.

`JUDGE_REPEATS` sets how often to score each saved answer. An odd count gives one middle score, the **median**.

**You should see** the index, semantic configuration, answer model, evaluator model and judge repeat count.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
from statistics import median
from uuid import uuid4

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper_path = folder / "labs" / "day_2" / "evaluation_utils.py"
    if helper_path.exists():
        REPO_ROOT = folder
        sys.path.insert(0, str(helper_path.parent))
        break
else:
    raise FileNotFoundError("labs/day_2/evaluation_utils.py was not found.")

from evaluation_utils import (
    answer_json_schema,
    context_signal_coverage,
    document_evidence_keys,
    estimate_cost_usd,
    format_search_context,
    load_cases,
    price_rates_from_env,
    primitive,
    ranked_evidence_metrics,
    render_cited_answer,
    response_token_usage,
    retrieval_recall,
    validate_cited_answer,
)
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from azure.search.documents import SearchClient

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
EVALUATOR_MODEL = os.getenv("RAG_EVALUATOR_MODEL", "") or MODEL_DEPLOYMENT
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX_NAME", "")
SEMANTIC_CONFIGURATION = os.getenv(
    "AZURE_SEARCH_SEMANTIC_CONFIGURATION", "who-guidelines-semantic"
)
JUDGE_REPEATS = int(os.getenv("RAG_EVAL_JUDGE_REPEATS", "3"))
if not 3 <= JUDGE_REPEATS <= 9 or JUDGE_REPEATS % 2 == 0:
    raise ValueError("RAG_EVAL_JUDGE_REPEATS must be an odd number from 3 to 9.")
missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
        "AZURE_SEARCH_ENDPOINT": SEARCH_ENDPOINT,
        "AZURE_SEARCH_INDEX_NAME": SEARCH_INDEX,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


def wait_for_run(eval_id, run, expected_items):
    deadline = time.monotonic() + 1200
    while run.status not in ("completed", "failed", "canceled"):
        if time.monotonic() > deadline:
            raise TimeoutError("Evaluation exceeded 20 minutes.")
        time.sleep(5)
        run = client.evals.runs.retrieve(run_id=run.id, eval_id=eval_id)
        print("status:", run.status)
    items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    if run.status != "completed":
        raise RuntimeError(f"Evaluation ended as {run.status}: {primitive(getattr(run, 'error', None))}")
    item_deadline = time.monotonic() + 120
    while len(items) < expected_items:
        if time.monotonic() > item_deadline:
            raise TimeoutError(f"Only {len(items)}/{expected_items} output items became visible.")
        time.sleep(2)
        items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    return run, items


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, credential=credential)
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)
RATES = price_rates_from_env()
print({
    "index": SEARCH_INDEX,
    "semantic_configuration": SEMANTIC_CONFIGURATION,
    "generator": MODEL_DEPLOYMENT,
    "judge": EVALUATOR_MODEL,
    "judge_repeats": JUDGE_REPEATS,
})

## 1. Load the benchmark

A **benchmark** is a fixed set of questions and expectations. This one covers IPC staffing, hypertension treatment and type 2 diabetes diagnosis.

| Field | What it describes |
|---|---|
| `expected_evidence_groups` | Publications Search should find |
| `required_context_signals` | Words or phrases to look for in retrieved passages |
| `required_answer_parts` | Parts a complete answer must cover |
| `ground_truth` | A reference answer used only for evaluation |

These answer-part and citation rules form the **answer contract**. The model receives the required parts, but not the reference answer, expected evidence labels or context signals.

Finding a publication or keyword does not prove every needed fact was retrieved. The later checks also examine the answer's completeness.

The dataset version keeps expectations consistent across runs.

**You should see** dataset and answer-contract versions plus three case IDs.

In [ ]:
DATASET_PATH = REPO_ROOT / "labs" / "data" / "medical_rag_evaluation_cases.json"
DATASET, SEARCH_CASES = load_cases(DATASET_PATH, "azure_ai_search")
assert DATASET["dataset_id"] == "umc-who-rag-evaluation-v1"
assert len(SEARCH_CASES) == 3
print({
    "dataset": DATASET["dataset_id"],
    "answer_contract": DATASET["answer_contract_version"],
    "cases": [case["case_id"] for case in SEARCH_CASES],
})

### To-Do 1 - Choose how many results to inspect

Set two limits:

- `TOP_K = 5`: score the original top five results without reordering them.
- `CANDIDATE_CAP = 10`: consider up to ten results when choosing passages for the answer.

**Recall@k** measures how much expected evidence appears in the first `k` results. **Precision@k** measures how much of that set is expected evidence.

More candidates may recover useful passages, but can also add irrelevant text. Keep the original ranking visible rather than hiding weak results in a larger pool.

**Predict:** what does low Recall@5 but high recall at 10 tell you?

<details><summary>Show solution code</summary>

```python
TOP_K = 5
CANDIDATE_CAP = 10
```

</details>

In [ ]:
TOP_K = ...  # TODO 1: rank depth used for headline retrieval quality.
CANDIDATE_CAP = ...  # TODO 1: bounded depth available for coherent generation context.
check_todos(TOP_K=TOP_K, CANDIDATE_CAP=CANDIDATE_CAP)
assert TOP_K == 5
assert CANDIDATE_CAP == 10
print({"top_k": TOP_K, "candidate_cap": CANDIDATE_CAP})

## 2. Retrieve evidence and generate answers

The next two cells define the helpers, then run each case:

1. Get up to ten semantically ranked passages and score the top five.
2. Keep candidates from the top result's `publication_id`, avoiding a mix of different guidelines.
3. Check the selected text for context signals and label sources `[S1]`, `[S2]`, and so on.
4. Ask the model for JSON claims linked to required answer parts and supplied source IDs.
5. Check answer coverage and citations; record time, tokens and Search requests.

The selected text is the **generation context**: exactly what the answer model sees. Unsupported answer parts must be marked `insufficient_evidence`, not filled from memory.

The report also shows **reciprocal rank**: the first expected result scores 1 at rank 1, 0.5 at rank 2, and so on.

**You should see** three records with cited answers, followed by `PASS`.

In [ ]:
def generate_cited_answer(case, context, valid_source_ids):
    required_parts = case["required_answer_parts"]
    part_ids = [part["id"] for part in required_parts]
    parts_text = "\n".join(
        f"- {part['id']}: {part['description']}" for part in required_parts
    )
    schema = answer_json_schema(part_ids, valid_source_ids)
    response = client.responses.create(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "Answer only from the supplied WHO Search sources. Cover each required answer part "
            "with one or more concise factual claims. Give every claim exactly one answer_part_id "
            "and one or more supporting source_ids. Put a part in insufficient_evidence only when "
            "the supplied sources cannot support it, and never both answer and mark the same part "
            "insufficient. Do not add patient-specific advice or use source IDs outside the schema."
        ),
        input=(
            f"Question: {case['query']}\n\n"
            f"Required answer parts ({DATASET['answer_contract_version']}):\n{parts_text}\n\n"
            f"Search sources:\n{context}"
        ),
        text={
            "format": {
                "type": "json_schema",
                "name": "medical_search_answer",
                "strict": True,
                "schema": schema,
            }
        },
        max_output_tokens=1400,
    )
    validation = validate_cited_answer(
        json.loads(response.output_text),
        valid_source_ids,
        part_ids,
    )
    return response, validation, render_cited_answer(validation)


print("Cited-answer generator ready.")

In [ ]:
SEARCH_ROWS = []
for case in SEARCH_CASES:
    total_started = time.perf_counter()
    retrieval_started = time.perf_counter()
    documents = [
        dict(result)
        for result in search_client.search(
            search_text=case["query"],
            query_type="semantic",
            semantic_configuration_name=SEMANTIC_CONFIGURATION,
            top=CANDIDATE_CAP,
            select=[
                "chunk_id",
                "parent_id",
                "chunk",
                "document_title",
                "source_url",
                "publication_id",
                "topic",
                "metadata_storage_name",
            ],
        )
    ]
    retrieval_ms = (time.perf_counter() - retrieval_started) * 1000
    if not documents:
        raise RuntimeError(f"{case['case_id']}: Search returned no documents.")

    top_documents = documents[:TOP_K]
    top_keys = set().union(*(document_evidence_keys(doc) for doc in top_documents))
    candidate_keys = set().union(*(document_evidence_keys(doc) for doc in documents))
    recall_at_k = retrieval_recall(case["expected_evidence_groups"], top_keys)
    candidate_recall = retrieval_recall(case["expected_evidence_groups"], candidate_keys)
    ranking = ranked_evidence_metrics(case["expected_evidence_groups"], top_documents)

    primary_publication = str(documents[0].get("publication_id") or "").strip()
    if not primary_publication:
        raise RuntimeError(f"{case['case_id']}: the top result has no publication_id.")
    generation_documents = [
        document
        for document in documents
        if str(document.get("publication_id") or "").strip() == primary_publication
    ]
    context = format_search_context(generation_documents)
    signal_result = context_signal_coverage(case["required_context_signals"], context)
    generation_keys = set().union(
        *(document_evidence_keys(doc) for doc in generation_documents)
    )
    generation_document_recall = retrieval_recall(
        case["expected_evidence_groups"], generation_keys
    )
    valid_source_ids = [f"S{index}" for index in range(1, len(generation_documents) + 1)]

    generation_started = time.perf_counter()
    response, citations, answer = generate_cited_answer(case, context, valid_source_ids)
    generation_ms = (time.perf_counter() - generation_started) * 1000
    usage = response_token_usage(response, semantic_requests=1)
    cost = estimate_cost_usd(usage, RATES)

    row = {
        "case_id": case["case_id"],
        "query": case["query"],
        "ground_truth": case["ground_truth"],
        "context": context,
        "response": answer,
        "raw_sources": [
            {
                "rank": rank,
                "publication_id": document.get("publication_id"),
                "chunk_id": document.get("chunk_id"),
                "reranker_score": document.get("@search.reranker_score"),
            }
            for rank, document in enumerate(documents, start=1)
        ],
        "primary_publication": primary_publication,
        "retrieval_recall_at_5": recall_at_k["recall"],
        "candidate_recall_at_10": candidate_recall["recall"],
        "precision_at_5": ranking["precision"],
        "top_1_match": ranking["top_1_match"],
        "first_match_rank": ranking["first_match_rank"],
        "reciprocal_rank": ranking["reciprocal_rank"],
        "generation_document_recall": generation_document_recall["recall"],
        "context_signal_coverage": signal_result["coverage"],
        "context_signal_details": signal_result["signals"],
        "answer_part_coverage": citations["answer_part_coverage"],
        "insufficient_evidence": citations["insufficient_evidence"],
        "citation_coverage": citations["citation_coverage"],
        "citation_validity": citations["citation_validity"],
        "retrieval_ms": round(retrieval_ms, 2),
        "generation_ms": round(generation_ms, 2),
        "total_ms": round((time.perf_counter() - total_started) * 1000, 2),
        **usage,
        **cost,
    }
    SEARCH_ROWS.append(row)
    print(json.dumps({
        key: row[key]
        for key in (
            "case_id",
            "primary_publication",
            "raw_sources",
            "retrieval_recall_at_5",
            "candidate_recall_at_10",
            "precision_at_5",
            "top_1_match",
            "reciprocal_rank",
            "generation_document_recall",
            "context_signal_coverage",
            "answer_part_coverage",
            "citation_coverage",
            "citation_validity",
            "retrieval_ms",
            "generation_ms",
            "model_input_tokens",
            "model_output_tokens",
            "estimated_cost_usd",
        )
    }, indent=2))
    print(answer, "\n")

assert len(SEARCH_ROWS) == len(SEARCH_CASES)
print("PASS - every benchmark case completed live retrieval and strictly cited generation.")

## 3. Score each saved answer several times

Two Foundry evaluators check answer quality:

- **Groundedness:** does the answer agree with the exact passages it used?
- **Response Completeness:** does it cover the independent reference answer?

Keep each answer unchanged while scoring it `JUDGE_REPEATS` times. This is a **frozen answer**: differences now come from the evaluator, not new model responses.

Keep every score and reason, then take the median. Do not regenerate answers until one happens to pass. Three cases and three judge repeats produce nine evaluation items.

Evaluator token use is reported separately from retrieval and answer generation.

**You should see** nine items, their scores and reasons, median scores and a report URL.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

eval_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "eval_item_id": {"type": "string"},
            "run_case_id": {"type": "string"},
            "judge_repeat": {"type": "integer"},
            "query": {"type": "string"},
            "context": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": [
            "eval_item_id",
            "run_case_id",
            "judge_repeat",
            "query",
            "context",
            "response",
            "ground_truth",
        ],
        "additionalProperties": False,
    },
    include_sample_schema=True,
)
criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "query": "{{item.query}}",
            "context": "{{item.context}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="response_completeness",
        evaluator_name="builtin.response_completeness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "ground_truth": "{{item.ground_truth}}",
            "response": "{{item.response}}",
        },
    ),
]
evaluation = client.evals.create(
    name=f"day2-search-rag-eval-{SUFFIX}",
    data_source_config=eval_config,
    testing_criteria=criteria,
)
eval_items = [
    {
        "item": {
            "eval_item_id": f"{row['case_id']}-J{repeat}",
            "run_case_id": row["case_id"],
            "judge_repeat": repeat,
            **{key: row[key] for key in ("query", "context", "response", "ground_truth")},
        }
    }
    for row in SEARCH_ROWS
    for repeat in range(1, JUDGE_REPEATS + 1)
]
eval_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day2-search-rag-run-{SUFFIX}",
    metadata={
        "dataset": DATASET["dataset_id"],
        "answer_contract": DATASET["answer_contract_version"],
        "target": "azure-ai-search",
        "judge_repeats": str(JUDGE_REPEATS),
    },
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": eval_items},
    },
)
print({"evaluation_id": evaluation.id, "run_id": eval_run.id, "items": len(eval_items)})

In [ ]:
eval_run, output_items = wait_for_run(evaluation.id, eval_run, len(eval_items))
METRICS = ("groundedness", "response_completeness")
scores = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in SEARCH_ROWS
}
reasons = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in SEARCH_ROWS
}
for item in output_items:
    data = primitive(item)
    source = data["datasource_item"]
    case_id = source["run_case_id"]
    repeat = int(source["judge_repeat"])
    for metric in METRICS:
        result = next(result for result in data["results"] if result["name"] == metric)
        if result.get("error") or result.get("status") in ("failed", "error", "canceled"):
            raise RuntimeError(f"{case_id}-J{repeat} {metric} failed: {result}")
        reason = str(result.get("reason") or "").strip()
        if not reason:
            raise RuntimeError(f"{case_id}-J{repeat} {metric} returned no reason.")
        scores[case_id][metric][repeat] = float(result["score"])
        reasons[case_id][metric][repeat] = reason

for row in SEARCH_ROWS:
    for metric in METRICS:
        samples = [scores[row["case_id"]][metric][repeat] for repeat in range(1, JUDGE_REPEATS + 1)]
        row[f"{metric}_samples"] = samples
        row[f"{metric}_reasons"] = [
            reasons[row["case_id"]][metric][repeat]
            for repeat in range(1, JUDGE_REPEATS + 1)
        ]
        row[f"{metric}_mean"] = round(sum(samples) / len(samples), 3)
        row[metric] = float(median(samples))

print({
    "consensus": {
        row["case_id"]: {
            metric: {
                "samples": row[f"{metric}_samples"],
                "median": row[metric],
                "reasons": row[f"{metric}_reasons"],
            }
            for metric in METRICS
        }
        for row in SEARCH_ROWS
    },
    "evaluator_usage": [primitive(item) for item in (getattr(eval_run, "per_model_usage", None) or [])],
    "report_url": getattr(eval_run, "report_url", None),
})

### To-Do 2 - Set the passing scores

Before viewing the decisions, set `MIN_GROUNDEDNESS` and `MIN_COMPLETENESS` to `4.0` on the 1-to-5 scale.

These checks form the **release gate**. A case gets `ship` only when all pass:

- The expected publication is first and fully represented in both the candidate set and selected context.
- All context signals and required answer parts are covered; no part is marked unsupported.
- Every claim cites a supplied source.
- Median Groundedness and Response Completeness are each at least 4.

The first three are exact Python checks; the last uses model judgement. A high model score cannot excuse a missing part or invalid citation. These are workshop thresholds, not clinical standards.

<details><summary>Show solution code</summary>

```python
MIN_GROUNDEDNESS = 4.0
MIN_COMPLETENESS = 4.0
```

</details>

In [ ]:
MIN_GROUNDEDNESS = ...  # TODO 2: median groundedness threshold on the 1-to-5 scale.
MIN_COMPLETENESS = ...  # TODO 2: median completeness threshold on the 1-to-5 scale.
check_todos(MIN_GROUNDEDNESS=MIN_GROUNDEDNESS, MIN_COMPLETENESS=MIN_COMPLETENESS)
assert MIN_GROUNDEDNESS == MIN_COMPLETENESS == 4.0

for row in SEARCH_ROWS:
    row["deterministic_gate_passed"] = all([
        row["candidate_recall_at_10"] == 1.0,
        row["generation_document_recall"] == 1.0,
        row["context_signal_coverage"] == 1.0,
        row["top_1_match"],
        row["answer_part_coverage"] == 1.0,
        not row["insufficient_evidence"],
        row["citation_coverage"] == 1.0,
        row["citation_validity"] == 1.0,
    ])
    row["semantic_gate_passed"] = (
        row["groundedness"] >= MIN_GROUNDEDNESS
        and row["response_completeness"] >= MIN_COMPLETENESS
    )
    row["release_gate_passed"] = (
        row["deterministic_gate_passed"] and row["semantic_gate_passed"]
    )
    row["decision"] = "ship" if row["release_gate_passed"] else "mitigate"
    print(json.dumps({
        "case_id": row["case_id"],
        "decision": row["decision"],
        "retrieval": {
            "recall_at_5": row["retrieval_recall_at_5"],
            "candidate_recall_at_10": row["candidate_recall_at_10"],
            "precision_at_5": row["precision_at_5"],
            "top_1_match": row["top_1_match"],
            "reciprocal_rank": row["reciprocal_rank"],
        },
        "context_signal_coverage": row["context_signal_coverage"],
        "answer_part_coverage": row["answer_part_coverage"],
        "citation_coverage": row["citation_coverage"],
        "citation_validity": row["citation_validity"],
        "groundedness": {
            "samples": row["groundedness_samples"],
            "median": row["groundedness"],
            "reasons": row["groundedness_reasons"],
        },
        "response_completeness": {
            "samples": row["response_completeness_samples"],
            "median": row["response_completeness"],
            "reasons": row["response_completeness_reasons"],
        },
        "latency_ms": {
            "retrieval": row["retrieval_ms"],
            "generation": row["generation_ms"],
            "total": row["total_ms"],
        },
        "usage": {
            "model_input_tokens": row["model_input_tokens"],
            "model_output_tokens": row["model_output_tokens"],
            "semantic_requests": row["semantic_requests"],
        },
        "estimated_cost_usd": row["estimated_cost_usd"],
    }, indent=2))

## Confirm that every case produced a report

The final check requires all three cases to return retrieval and citation metrics, repeated scores and reasons, timing, usage, and a decision.

`PASS` means the evaluation completed, not that every answer should ship. A `mitigate` decision is valid evidence of work still needed.

**You should see** `PASS` and one decision per case.

In [ ]:
assert eval_run.status == "completed"
assert len(SEARCH_ROWS) == len(SEARCH_CASES) == 3
assert len(output_items) == len(SEARCH_ROWS) * JUDGE_REPEATS
assert all(0.0 <= row["retrieval_recall_at_5"] <= 1.0 for row in SEARCH_ROWS)
assert all(0.0 <= row["precision_at_5"] <= 1.0 for row in SEARCH_ROWS)
assert all(0.0 <= row["citation_validity"] <= 1.0 for row in SEARCH_ROWS)
assert all(len(row["groundedness_samples"]) == JUDGE_REPEATS for row in SEARCH_ROWS)
assert all(len(row["response_completeness_reasons"]) == JUDGE_REPEATS for row in SEARCH_ROWS)
assert all(row["retrieval_ms"] > 0 and row["total_ms"] >= row["retrieval_ms"] for row in SEARCH_ROWS)
assert all(row["decision"] in {"ship", "mitigate"} for row in SEARCH_ROWS)
print("PASS - retrieval, context, generation, citation, judge, latency, usage and release evidence are populated.")
print("Decisions:", {row["case_id"]: row["decision"] for row in SEARCH_ROWS})

## Find the first failing step

| Failure | Inspect |
|---|---|
| Expected publication missing or ranked low | Search query and ranking |
| Publication found but context signals missing | Selected passages and candidate depth |
| Evidence is sufficient but answer parts are missing | Generation instructions and required parts |
| Citation missing or unknown | JSON output and source labels |
| Exact checks pass but model score is low | Judge reason, answer and evidence |
| Quality passes but time or usage is high | Retrieval and model-call costs |

Investigate the failure rather than lowering a threshold to make it pass.

<details><summary>Optional cost estimates</summary>

Token and Search request counts are always reported. USD estimates need rates from your applicable price sheet; they are not Azure billing records.

</details>

## What you learned

- Check retrieval, selected passages and the answer separately.
- Valid source IDs do not prove a claim is supported; inspect quality scores and reasons too.
- Score the same answer repeatedly, then combine quality, timing and usage in your decision.

**Check your understanding**

1. Candidate recall is 1.0 but context-signal coverage is 0.5. What needs investigation?
2. Citation IDs are valid but Groundedness is low. What do valid IDs not prove?
3. Why are three cases insufficient for a production reliability claim?

<details><summary>Compare your answers</summary>

1. The publication was found, but some expected phrases are missing from the selected passages. Inspect those passages for missing facts.
2. They identify supplied sources, not whether claims accurately represent them.
3. They do not cover real query frequency, languages, edge cases or quality variation.

</details>

Further reading: [semantic ranking](https://learn.microsoft.com/azure/search/semantic-how-to-query-request), [RAG evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rag-evaluators), and [RAG evaluation design](https://learn.microsoft.com/azure/architecture/ai-ml/guide/rag/rag-solution-design-and-evaluation-guide).

**Expected artifact:** three case reports with retrieval, context, citation, judge, timing, usage and decision details.

**Finish:** close local clients. The report remains in Foundry.

**Next:** Lab 13 uses Foundry IQ and repeats the live path.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed local clients. The evaluation report remains in Foundry.")